# Module 09: Multi-Agent Orchestration

**Companion notebook for**: `09-multi-agent.html`

## Learning Objectives
- Define specialized agent roles (researcher, coder, reviewer)
- Build an orchestrator that delegates tasks to the right agent
- Implement message passing between agents
- Synthesize results from multiple agents into a final output
- Control execution with max iteration limits

## Prerequisites
- Python 3.10+
- `ANTHROPIC_API_KEY` environment variable set (optional -- notebook uses mock fallbacks)
- Familiarity with basic agent concepts (see Module 06)

---
## Setup & Dependencies

In [ ]:
%pip install -q anthropic pydantic

In [ ]:
import os
import json
import time
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum

API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
USE_LIVE_API = bool(API_KEY)

print(f"Live API available: {USE_LIVE_API}")
if not USE_LIVE_API:
    print("Running in mock/simulation mode -- all outputs are simulated.")

---
## Section 1: Define Agent Roles

Each agent has a specific role, system prompt, and capabilities.
We define three agents: Researcher, Coder, and Reviewer.

In [ ]:
class AgentRole(str, Enum):
    RESEARCHER = "researcher"
    CODER = "coder"
    REVIEWER = "reviewer"
    ORCHESTRATOR = "orchestrator"


@dataclass
class Message:
    """A message passed between agents."""
    sender: str
    recipient: str
    content: str
    message_type: str = "task"  # task, result, feedback, final
    metadata: dict = field(default_factory=dict)


@dataclass
class AgentConfig:
    """Configuration for an agent."""
    name: str
    role: AgentRole
    system_prompt: str
    capabilities: list[str]
    max_retries: int = 2


AGENT_CONFIGS = {
    AgentRole.RESEARCHER: AgentConfig(
        name="ResearchBot",
        role=AgentRole.RESEARCHER,
        system_prompt=(
            "You are a research specialist. Your job is to gather information, "
            "analyze requirements, identify relevant technologies, and provide "
            "structured research summaries. Always cite sources and highlight "
            "trade-offs between approaches."
        ),
        capabilities=["information_gathering", "technology_analysis", "requirements_analysis"],
    ),
    AgentRole.CODER: AgentConfig(
        name="CodeBot",
        role=AgentRole.CODER,
        system_prompt=(
            "You are a coding specialist. Your job is to write clean, "
            "well-documented, production-ready code based on research findings "
            "and specifications. Include type hints, error handling, and docstrings."
        ),
        capabilities=["code_generation", "code_refactoring", "testing"],
    ),
    AgentRole.REVIEWER: AgentConfig(
        name="ReviewBot",
        role=AgentRole.REVIEWER,
        system_prompt=(
            "You are a code review specialist. Your job is to review code for "
            "correctness, security, performance, and best practices. Provide "
            "specific, actionable feedback with severity levels (critical, major, minor)."
        ),
        capabilities=["code_review", "security_analysis", "best_practices"],
    ),
}

print("Agent Roles Defined")
print("=" * 60)
for role, config in AGENT_CONFIGS.items():
    print(f"\n  {config.name} ({role.value})")
    print(f"  Capabilities: {', '.join(config.capabilities)}")
    print(f"  Prompt: {config.system_prompt[:80]}...")

---
## Section 2: Individual Agent Implementation

Each agent processes incoming messages and produces a response.
In production, each agent calls the LLM with its specific system prompt.

In [ ]:
class Agent:
    """An individual agent that processes messages."""
    
    def __init__(self, config: AgentConfig):
        self.config = config
        self.message_history: list[Message] = []
    
    def process(self, message: Message) -> Message:
        """Process an incoming message and produce a response."""
        self.message_history.append(message)
        
        # Try live API first
        response_content = self._call_llm(message.content)
        if response_content is None:
            response_content = self._simulate_response(message.content)
        
        response = Message(
            sender=self.config.name,
            recipient=message.sender,
            content=response_content,
            message_type="result",
        )
        self.message_history.append(response)
        return response
    
    def _call_llm(self, prompt: str) -> Optional[str]:
        """Call the LLM API. Returns None if unavailable."""
        if not USE_LIVE_API:
            return None
        try:
            import anthropic
            client = anthropic.Anthropic(api_key=API_KEY)
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=1024,
                system=self.config.system_prompt,
                messages=[{"role": "user", "content": prompt}],
            )
            return response.content[0].text
        except Exception:
            return None
    
    def _simulate_response(self, prompt: str) -> str:
        """Simulate agent response based on role."""
        role = self.config.role
        
        if role == AgentRole.RESEARCHER:
            return (
                "## Research Findings\n\n"
                "### Requirements Analysis\n"
                "- The task requires a REST API endpoint for text classification\n"
                "- Key technologies: FastAPI (performance), Pydantic (validation)\n"
                "- Estimated complexity: Medium\n\n"
                "### Recommended Approach\n"
                "1. Use FastAPI for the web framework (async support, auto-docs)\n"
                "2. Pydantic models for request/response validation\n"
                "3. In-memory caching for repeated queries\n\n"
                "### Trade-offs\n"
                "- FastAPI vs Flask: FastAPI is faster but has a steeper learning curve\n"
                "- Sync vs Async: Async is better for I/O-bound LLM calls\n"
            )
        elif role == AgentRole.CODER:
            return (
                '```python\n'
                'from fastapi import FastAPI, HTTPException\n'
                'from pydantic import BaseModel, Field\n'
                'from typing import Literal\n\n'
                'app = FastAPI(title="Text Classifier API")\n\n'
                'class ClassifyRequest(BaseModel):\n'
                '    text: str = Field(min_length=1, max_length=5000)\n\n'
                'class ClassifyResponse(BaseModel):\n'
                '    category: str\n'
                '    confidence: float = Field(ge=0.0, le=1.0)\n\n'
                '@app.post("/classify", response_model=ClassifyResponse)\n'
                'async def classify(request: ClassifyRequest):\n'
                '    """Classify input text into a category."""\n'
                '    try:\n'
                '        result = await run_classifier(request.text)\n'
                '        return ClassifyResponse(\n'
                '            category=result["category"],\n'
                '            confidence=result["confidence"],\n'
                '        )\n'
                '    except Exception as e:\n'
                '        raise HTTPException(status_code=500, detail=str(e))\n'
                '```\n'
            )
        elif role == AgentRole.REVIEWER:
            return (
                "## Code Review\n\n"
                "### Overall: APPROVE with minor changes\n\n"
                "### Findings\n"
                "1. [MINOR] Add rate limiting middleware to prevent abuse\n"
                "2. [MINOR] Add request ID for tracing/logging\n"
                "3. [MAJOR] `run_classifier` is not defined -- needs implementation\n"
                "4. [MINOR] Consider adding input sanitization before classification\n"
                "5. [MINOR] Add response caching for repeated queries\n\n"
                "### Security\n"
                "- No SQL injection risk (no database queries)\n"
                "- Input length validation is good (max 5000 chars)\n"
                "- Recommendation: Add authentication middleware\n"
            )
        return "Processed the request."


# Test individual agents
researcher = Agent(AGENT_CONFIGS[AgentRole.RESEARCHER])
test_msg = Message(
    sender="orchestrator",
    recipient="ResearchBot",
    content="Research how to build a text classification API.",
)
result = researcher.process(test_msg)
print(f"Agent: {result.sender}")
print(f"Response:\n{result.content[:300]}...")

---
## Section 3: Message Passing System

Agents communicate through a message bus. The message bus logs all messages
and routes them between agents.

In [ ]:
class MessageBus:
    """Central message bus for inter-agent communication."""
    
    def __init__(self):
        self.messages: list[Message] = []
        self.agents: dict[str, Agent] = {}
    
    def register_agent(self, agent: Agent):
        """Register an agent with the message bus."""
        self.agents[agent.config.name] = agent
    
    def send(self, message: Message) -> Optional[Message]:
        """Send a message to the recipient agent and get a response."""
        self.messages.append(message)
        
        recipient = self.agents.get(message.recipient)
        if recipient is None:
            error_msg = Message(
                sender="system",
                recipient=message.sender,
                content=f"Error: Unknown agent '{message.recipient}'",
                message_type="error",
            )
            self.messages.append(error_msg)
            return error_msg
        
        response = recipient.process(message)
        self.messages.append(response)
        return response
    
    def get_conversation_log(self) -> str:
        """Get a formatted log of all messages."""
        lines = []
        for i, msg in enumerate(self.messages):
            lines.append(f"[{i+1}] {msg.sender} -> {msg.recipient} [{msg.message_type}]")
            # Show first 100 chars of content
            content_preview = msg.content[:100].replace('\n', ' ')
            lines.append(f"    {content_preview}...")
        return "\n".join(lines)


# Test message passing
bus = MessageBus()
bus.register_agent(Agent(AGENT_CONFIGS[AgentRole.RESEARCHER]))
bus.register_agent(Agent(AGENT_CONFIGS[AgentRole.CODER]))
bus.register_agent(Agent(AGENT_CONFIGS[AgentRole.REVIEWER]))

# Send a message from orchestrator to researcher
msg = Message(
    sender="orchestrator",
    recipient="ResearchBot",
    content="Research best practices for building a text classifier API.",
    message_type="task",
)
response = bus.send(msg)
print(f"Response from {response.sender}:")
print(response.content[:200])

print(f"\n{'='*60}")
print("Message Log:")
print(bus.get_conversation_log())

---
## Section 4: Orchestrator

The orchestrator manages the workflow: it breaks down the user's request,
delegates subtasks to the right agents, and synthesizes the final result.

In [ ]:
@dataclass
class TaskPlan:
    """A plan of subtasks to execute."""
    original_request: str
    steps: list[dict]  # Each: {"agent": str, "task": str, "depends_on": list[int]}


class Orchestrator:
    """
    Multi-agent orchestrator that decomposes tasks, delegates to agents,
    and synthesizes results.
    """
    
    def __init__(self, message_bus: MessageBus, max_iterations: int = 10):
        self.bus = message_bus
        self.max_iterations = max_iterations
        self.results: dict[int, Message] = {}  # step_index -> result
    
    def plan(self, request: str) -> TaskPlan:
        """
        Decompose a user request into subtasks.
        In production, the LLM generates this plan.
        """
        # Simulated planning (an LLM would generate this)
        return TaskPlan(
            original_request=request,
            steps=[
                {
                    "agent": "ResearchBot",
                    "task": f"Research the best approach for: {request}",
                    "depends_on": [],
                },
                {
                    "agent": "CodeBot",
                    "task": "Based on the research findings, write the implementation code.",
                    "depends_on": [0],
                },
                {
                    "agent": "ReviewBot",
                    "task": "Review the code for correctness, security, and best practices.",
                    "depends_on": [1],
                },
            ],
        )
    
    def execute(self, request: str) -> dict:
        """Execute the full multi-agent workflow."""
        plan = self.plan(request)
        
        print(f"Orchestrator: Received request")
        print(f"  '{request}'")
        print(f"  Planned {len(plan.steps)} steps, max iterations: {self.max_iterations}")
        print("=" * 60)
        
        iteration = 0
        
        for step_idx, step in enumerate(plan.steps):
            iteration += 1
            if iteration > self.max_iterations:
                print(f"\nMax iterations ({self.max_iterations}) reached. Stopping.")
                break
            
            agent_name = step["agent"]
            task = step["task"]
            
            # Build context from dependencies
            context_parts = [task]
            for dep_idx in step["depends_on"]:
                if dep_idx in self.results:
                    dep_result = self.results[dep_idx]
                    context_parts.append(
                        f"\n--- Previous result from {dep_result.sender} ---\n"
                        f"{dep_result.content}"
                    )
            
            full_task = "\n".join(context_parts)
            
            print(f"\n--- Step {step_idx + 1}: {agent_name} ---")
            print(f"  Task: {task[:80]}")
            if step["depends_on"]:
                print(f"  Dependencies: steps {step['depends_on']}")
            
            # Send to agent
            msg = Message(
                sender="orchestrator",
                recipient=agent_name,
                content=full_task,
                message_type="task",
                metadata={"step_index": step_idx},
            )
            response = self.bus.send(msg)
            self.results[step_idx] = response
            
            # Show abbreviated result
            preview = response.content[:150].replace('\n', ' ')
            print(f"  Result: {preview}...")
        
        # Synthesize final output
        final = self._synthesize(plan, self.results)
        return final
    
    def _synthesize(self, plan: TaskPlan, results: dict[int, Message]) -> dict:
        """Combine all agent results into a final deliverable."""
        sections = []
        for idx, step in enumerate(plan.steps):
            if idx in results:
                sections.append({
                    "agent": step["agent"],
                    "task": step["task"],
                    "output": results[idx].content,
                })
        
        return {
            "request": plan.original_request,
            "num_steps": len(sections),
            "sections": sections,
            "status": "complete" if len(sections) == len(plan.steps) else "partial",
        }


print("Orchestrator class defined.")
print("Methods: plan(request), execute(request), _synthesize(plan, results)")

---
## Section 5: Run the Multi-Agent Workflow

Execute a complete multi-agent workflow: the user makes a request,
the orchestrator decomposes it, agents collaborate, and results are synthesized.

In [ ]:
# Create fresh agents and message bus
bus = MessageBus()
bus.register_agent(Agent(AGENT_CONFIGS[AgentRole.RESEARCHER]))
bus.register_agent(Agent(AGENT_CONFIGS[AgentRole.CODER]))
bus.register_agent(Agent(AGENT_CONFIGS[AgentRole.REVIEWER]))

orchestrator = Orchestrator(bus, max_iterations=10)

# Execute the workflow
result = orchestrator.execute(
    "Build a text classification API with FastAPI that categorizes "
    "customer support messages into billing, technical, and account categories."
)

print("\n" + "=" * 60)
print("FINAL SYNTHESIZED OUTPUT")
print("=" * 60)
print(f"Status: {result['status']}")
print(f"Steps completed: {result['num_steps']}")
for section in result["sections"]:
    print(f"\n--- {section['agent']} ---")
    print(section["output"][:300])

---
## Section 6: Full Conversation Trace

Inspect the complete message trace to understand agent interactions.

In [ ]:
print("Complete Message Trace")
print("=" * 60)
print(bus.get_conversation_log())

print(f"\n\nMessage Statistics")
print("=" * 40)
print(f"  Total messages: {len(bus.messages)}")

# Count by sender
from collections import Counter
senders = Counter(m.sender for m in bus.messages)
print(f"  Messages by sender:")
for sender, count in senders.most_common():
    print(f"    {sender}: {count}")

# Count by type
types = Counter(m.message_type for m in bus.messages)
print(f"  Messages by type:")
for mtype, count in types.most_common():
    print(f"    {mtype}: {count}")

---
## Section 7: Advanced Pattern -- Iterative Refinement

In some workflows, the reviewer's feedback loops back to the coder
for revisions. This demonstrates a cyclic multi-agent pattern with
max iteration control to prevent infinite loops.

In [ ]:
class IterativeOrchestrator:
    """
    Orchestrator with coder-reviewer feedback loop.
    The reviewer can request revisions up to max_revisions times.
    """
    
    def __init__(self, message_bus: MessageBus, max_revisions: int = 2):
        self.bus = message_bus
        self.max_revisions = max_revisions
    
    def execute(self, request: str) -> dict:
        """Execute with iterative refinement."""
        print(f"Iterative Orchestrator (max {self.max_revisions} revisions)")
        print("=" * 60)
        
        # Step 1: Research
        print("\n[Phase 1] Research")
        research_msg = Message(
            sender="orchestrator", recipient="ResearchBot",
            content=f"Research: {request}", message_type="task",
        )
        research_result = self.bus.send(research_msg)
        print(f"  ResearchBot completed.")
        
        # Step 2: Code + Review loop
        code_context = (
            f"Implement based on research:\n{research_result.content}\n\n"
            f"Original request: {request}"
        )
        
        for revision in range(self.max_revisions + 1):
            # Code
            print(f"\n[Phase 2.{revision + 1}a] Coding (revision {revision})")
            code_msg = Message(
                sender="orchestrator", recipient="CodeBot",
                content=code_context, message_type="task",
            )
            code_result = self.bus.send(code_msg)
            print(f"  CodeBot completed.")
            
            # Review
            print(f"[Phase 2.{revision + 1}b] Review")
            review_msg = Message(
                sender="orchestrator", recipient="ReviewBot",
                content=f"Review this code:\n{code_result.content}",
                message_type="task",
            )
            review_result = self.bus.send(review_msg)
            print(f"  ReviewBot completed.")
            
            # Check if review approves
            if "APPROVE" in review_result.content.upper() and "MAJOR" not in review_result.content.upper():
                print(f"  Review: APPROVED (no major issues)")
                break
            else:
                if revision < self.max_revisions:
                    print(f"  Review: REVISIONS REQUESTED")
                    code_context = (
                        f"Revise your code based on this feedback:\n"
                        f"{review_result.content}\n\n"
                        f"Previous code:\n{code_result.content}"
                    )
                else:
                    print(f"  Max revisions reached. Accepting current version.")
        
        print(f"\n{'='*60}")
        print(f"Workflow complete. Total messages: {len(self.bus.messages)}")
        
        return {
            "research": research_result.content,
            "code": code_result.content,
            "review": review_result.content,
            "revisions": revision,
        }


# Run iterative workflow
bus2 = MessageBus()
bus2.register_agent(Agent(AGENT_CONFIGS[AgentRole.RESEARCHER]))
bus2.register_agent(Agent(AGENT_CONFIGS[AgentRole.CODER]))
bus2.register_agent(Agent(AGENT_CONFIGS[AgentRole.REVIEWER]))

iter_orch = IterativeOrchestrator(bus2, max_revisions=2)
result = iter_orch.execute("Build a rate-limited API endpoint with authentication.")

print(f"\nRevisions performed: {result['revisions']}")
print(f"Total messages exchanged: {len(bus2.messages)}")

---
## Summary

In this notebook we built a complete multi-agent orchestration system:

1. **Agent Roles** -- Three specialized agents (Researcher, Coder, Reviewer), each with a distinct system prompt and capabilities.

2. **Agent Implementation** -- Each agent processes messages using its role-specific LLM prompt, with mock fallbacks for offline use.

3. **Message Passing** -- A central MessageBus routes messages between agents, logging all communication for debugging.

4. **Orchestrator** -- Decomposes user requests into a task plan, delegates to agents respecting dependencies, and synthesizes final output.

5. **Full Workflow** -- End-to-end execution: Research -> Code -> Review, with context passing between steps.

6. **Conversation Trace** -- Complete audit trail of all inter-agent messages for debugging and evaluation.

7. **Iterative Refinement** -- Coder-Reviewer feedback loop with max revision control to prevent infinite loops.

**Key insight**: Multi-agent systems decompose complex tasks into specialized subtasks. The orchestrator is the critical component -- it determines task decomposition, agent selection, and when to stop iterating.

**Next notebook**: Module 10 covers production platform architecture.